# Convert Old Responses CSV → New Format

**รองรับ 2 รูปแบบ input:**
- **Format เก่ามาก (E–AD):** answer index ที่ผู้เล่นเลือก (0, 1, 2) → ต้องมี column `s2q1`…`s16q2` เป็น index
- **Format กลาง (มี sec columns เป็น 0/1):** `s2q1`…`s16q2` เป็น 1/0 แต่ `sec2`…`sec16` เป็น binary all-or-nothing

**Output (format ใหม่):**
- E–AD : 26 ข้อ — `1` = ถูก, `0` = ผิด  
- AE–AS: 15 ส่วน (sec2–sec16) — **จำนวนข้อถูกในส่วน** (0 ถึง max ของส่วน)  
- AT–AV: 3 หมวด (cat1–cat3) — `1` = ถูกทุกข้อในหมวด, `0` = มีผิด  
- AW: score, AX: totalQuestions

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ── คำตอบที่ถูกต้องของแต่ละข้อ (0-based index) ──────────────
QUESTIONS = [
    {'id': 's2q1',  'section': 2,  'category': 1, 'correctIndex': 0},
    {'id': 's2q2',  'section': 2,  'category': 1, 'correctIndex': 2},
    {'id': 's3q1',  'section': 3,  'category': 1, 'correctIndex': 1},
    {'id': 's3q2',  'section': 3,  'category': 1, 'correctIndex': 2},
    {'id': 's4q1',  'section': 4,  'category': 1, 'correctIndex': 0},
    {'id': 's4q2',  'section': 4,  'category': 1, 'correctIndex': 1},
    {'id': 's4q3',  'section': 4,  'category': 1, 'correctIndex': 1},
    {'id': 's5q1',  'section': 5,  'category': 1, 'correctIndex': 0},
    {'id': 's5q2',  'section': 5,  'category': 1, 'correctIndex': 0},
    {'id': 's5q3',  'section': 5,  'category': 1, 'correctIndex': 1},
    {'id': 's6q1',  'section': 6,  'category': 1, 'correctIndex': 0},
    {'id': 's7q1',  'section': 7,  'category': 1, 'correctIndex': 1},
    {'id': 's8q1',  'section': 8,  'category': 2, 'correctIndex': 1},
    {'id': 's8q2',  'section': 8,  'category': 2, 'correctIndex': 1},
    {'id': 's9q1',  'section': 9,  'category': 2, 'correctIndex': 2},
    {'id': 's10q1', 'section': 10, 'category': 3, 'correctIndex': 1},
    {'id': 's10q2', 'section': 10, 'category': 3, 'correctIndex': 0},
    {'id': 's11q1', 'section': 11, 'category': 3, 'correctIndex': 2},
    {'id': 's12q1', 'section': 12, 'category': 3, 'correctIndex': 2},
    {'id': 's12q2', 'section': 12, 'category': 3, 'correctIndex': 1},
    {'id': 's13q1', 'section': 13, 'category': 3, 'correctIndex': 0},
    {'id': 's13q2', 'section': 13, 'category': 3, 'correctIndex': 1},
    {'id': 's14q1', 'section': 14, 'category': 3, 'correctIndex': 2},
    {'id': 's15q1', 'section': 15, 'category': 3, 'correctIndex': 2},
    {'id': 's16q1', 'section': 16, 'category': 3, 'correctIndex': 0},
    {'id': 's16q2', 'section': 16, 'category': 3, 'correctIndex': 1},
]

Q_IDS      = [q['id'] for q in QUESTIONS]
CORRECT    = {q['id']: q['correctIndex'] for q in QUESTIONS}
SECTIONS   = list(range(2, 17))   # 2–16
CATEGORIES = [1, 2, 3]

SEC_QS  = {s: [q['id'] for q in QUESTIONS if q['section']  == s] for s in SECTIONS}
CAT_QS  = {c: [q['id'] for q in QUESTIONS if q['category'] == c] for c in CATEGORIES}

print('Questions per section:', {s: len(v) for s, v in SEC_QS.items()})
print('Questions per category:', {c: len(v) for c, v in CAT_QS.items()})

In [ ]:
# ── โหลด CSV ─────────────────────────────────────────────────
# วาง path ของไฟล์ CSV ที่ export จาก Google Sheets ที่นี่
CSV_PATH = 'Teen Club Live - responses.csv'

df_old = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df_old)} rows')
print('Columns:', list(df_old.columns))

# ตรวจ format: ถ้า s2q1 มีค่า > 1 แสดงว่าเป็น answer index (format เก่ามาก)
is_index_format = df_old[Q_IDS].max().max() > 1
print('Input format:', 'answer index (0/1/2)' if is_index_format else '1/0 correctness')
df_old.head(3)

In [ ]:
# ── แปลงเป็น format ใหม่ ──────────────────────────────────────
rows = []

for _, row in df_old.iterrows():
    new = {
        'timestamp':  row['timestamp'],
        'gender':     row['gender'],
        'ageGroup':   row['ageGroup'],
        'education':  row['education'],
    }

    # แปลง question columns → 1/0 correctness
    correctness = {}
    for qid in Q_IDS:
        val = row.get(qid, -1)
        try:
            val = int(val)
        except (ValueError, TypeError):
            val = -1

        if is_index_format:
            # format เก่า: val คือ answer index
            correctness[qid] = 1 if val == CORRECT[qid] else 0
        else:
            # format กลาง: val คือ 1/0 correctness แล้ว
            correctness[qid] = val if val in (0, 1) else 0

        new[qid] = correctness[qid]

    # section: จำนวนข้อถูกในส่วน (count ไม่ใช่ binary)
    for sec in SECTIONS:
        qs = SEC_QS[sec]
        new[f'sec{sec}'] = sum(correctness[q] for q in qs)

    # category: 1 = ถูกทุกข้อในหมวด
    for cat in CATEGORIES:
        qs = CAT_QS[cat]
        new[f'cat{cat}'] = 1 if all(correctness[q] == 1 for q in qs) else 0

    new['score']          = sum(correctness.values())
    new['totalQuestions'] = 26

    rows.append(new)

df_new = pd.DataFrame(rows)
print(f'Done — {len(df_new)} rows, {len(df_new.columns)} columns')
df_new.head(3)

In [ ]:
# ── ตรวจสอบ column order ให้ตรงกับ Sheets ──────────────────────
expected_cols = (
    ['timestamp', 'gender', 'ageGroup', 'education'] +
    Q_IDS +
    [f'sec{s}' for s in SECTIONS] +
    [f'cat{c}' for c in CATEGORIES] +
    ['score', 'totalQuestions']
)
df_new = df_new[expected_cols]

# ตรวจสอบ: sec columns ควรมีค่า > 1 (เพราะเป็น count)
print('Section max values:', df_new[[f'sec{s}' for s in SECTIONS]].max().to_dict())
print('Column order OK')

In [ ]:
# ── save CSV สำหรับ copy-paste ─────────────────────────────────
OUT_PATH = 'responses_converted.csv'
df_new.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')  # utf-8-sig ให้ Excel/Sheets อ่านภาษาไทยได้
print(f'Saved → {OUT_PATH}')
df_new